---
title: Real-Time Causal Copy Testing for Ads with TabPFN 3.5
description: Predict a new ad creative's CTR before it serves, estimate whether LLM-written copy causally beats human copy, and let an LLM rewrite inside data-derived limits with TabPFN ranking the results.
icon: bullhorn
cookbookTags:
- regression
- causality
- text
- agent
authors:
- name: jinseriouspark
---
# Real-time causal copy testing for ads

An ad team with eight new creatives and one budget has to decide which to fund. Today that decision is bought with impressions: serve everything, read the signal, scale the winner. This cookbook uses TabPFN 3.5 to make that decision *before* anything serves, and then asks the follow-up question every team is asking in 2026: does LLM-written copy actually perform better than human copy?

Three things TabPFN does make this work:

1. **In-context fitting.** The advertiser's history is loaded as context. No training run, no tuning. A new advertiser is a cold start the model is built for.
2. **Text as text.** Headlines and body copy go in as columns. There is no vectoriser in this notebook.
3. **A predictive distribution.** `output_type="full"` gives an uncertainty band per creative, which is what turns a point estimate into a budget decision.

Everything below runs offline on a gradient-boosted baseline if `TABPFN_TOKEN` is not set, so you can read the outputs first and plug the real model in second.

In [ ]:
%pip install -q "adlift[client] @ git+https://github.com/jinseriouspark/tabpfn_for_ads" tabulate

## Authenticate (optional)

Set `TABPFN_TOKEN` to use hosted TabPFN 3.5. On Colab, store it as a secret. Without it, the cookbook runs on the offline baseline and every result says so.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ.setdefault("TABPFN_TOKEN", userdata.get("TABPFN_TOKEN") or "")
except Exception:
    pass

from adlift.model import resolve_backend
BACKEND = resolve_backend("auto")
print("model backend:", BACKEND)

## An ad account with a known answer

Real creative-level ad data with copy attached is proprietary. The generator below writes **both potential copies** for every creative, one in a human house style and one in an LLM house style, computes the click-through rate under each, and keeps both in hidden `_truth_*` columns. So the causal effect is known exactly and every estimate can be scored against it.

It also plants the three things that make this hard in practice: LLM adoption rises over time and clusters by vertical (confounding), the author changes copy length, tone and numeric claims (mediation), and creatives nest in campaigns that share a budget (grouping).

In [ ]:
import pandas as pd
from adlift.datasets.synth import make_ad_account, true_ate

account = make_ad_account(n_campaigns=120, seed=7)
print(f"{len(account)} creatives, {account.campaign_id.nunique()} campaigns")
print(f"planted total effect of LLM authorship: {true_ate(account, 'total') * 100:+.3f} pp")
print(f"planted direct effect (copy shape held fixed): {true_ate(account, 'direct') * 100:+.3f} pp")

with pd.option_context("display.max_colwidth", 60):
    display(account[["campaign_id", "author", "headline", "device", "week_index", "ctr"]].head(6))

The naive comparison is the pivot table everyone starts with:

In [ ]:
naive = account[account.author == "llm"].ctr.mean() - account[account.author == "human"].ctr.mean()
print(f"mean CTR(llm) - mean CTR(human) = {naive * 100:+.3f} pp")
print("LLM share of creatives by quarter of the period:")
print(account.assign(q=pd.cut(account.week_index, 4, labels=["Q1", "Q2", "Q3", "Q4"]))
             .groupby("q", observed=True).author.apply(lambda s: (s == "llm").mean()).round(2).to_string())

## Cold start: rank creatives in campaigns the model has never seen

Whole campaigns are held out with `GroupKFold`. Metrics are computed the way the budget is spent: how much CTR you give up by scaling the model's first pick, how often the true winner is in its top two, and how much of the exploration budget goes unspent if you only test the top two.

In [ ]:
from adlift.coldstart import comparison_table, evaluate_cold_start, random_baseline
from adlift.model import CTRModel

results = [random_baseline(account)]
results.append(evaluate_cold_start(account, CTRModel(backend="baseline"), name="GBDT + TF-IDF"))
if BACKEND != "baseline":
    results.append(evaluate_cold_start(account, CTRModel(backend=BACKEND), name="TabPFN 3.5"))
display(comparison_table(results))

## Does LLM copy causally outperform human copy?

One model per author is fitted on the **confounders** (placement, device, vertical, audience, budget, week) and every creative is scored under both arms; the mean gap is the total effect, with an interval from a cluster bootstrap that refits both arms. An S-learner (one model with the author flag) cross-checks it, a within-campaign placebo shuffle checks that the pipeline reports nothing when there is nothing, and an overlap diagnostic checks that the counterfactual question has an answer in this data.

Creative attributes (length, tone, numeric claim) are deliberately **not** in the adjustment set. The author determines them, so they are mediators; adjusting for them removes part of the effect being measured.

In [ ]:
from adlift.causal import full_analysis

analysis = full_analysis(account, model_factory=lambda: CTRModel(backend=BACKEND), n_boot=300)
total, direct, cross = analysis["total_effect"], analysis["direct_effect"], analysis["cross_check"]

print(f"naive difference        {analysis['naive_difference'] * 100:+.3f} pp")
print(f"T-learner total effect  {total.ate * 100:+.3f} pp  [{total.ci_low * 100:+.3f}, {total.ci_high * 100:+.3f}]  (headline, refit bootstrap)")
print(f"S-learner total effect  {cross.ate * 100:+.3f} pp  (cross-check)")
print(f"planted truth (total)   {true_ate(account, 'total') * 100:+.3f} pp")
print()
print(f"estimators agree: {analysis['estimators_agree']}")
print(f"placebo ratio:    {analysis['placebo']['ratio']:.1f}x")
print(f"overlap AUC:      {analysis['overlap']['auc']:.2f}, off support {analysis['overlap']['share_off_support'] * 100:.1f}%")
print(f"share of effect via visible copy attributes: {analysis['mediation']['indirect_share'] * 100:.0f}%")
display(analysis["segments"].round(5))

The pivot table and the adjusted estimate disagree on the **sign**. LLM copy was adopted later, in verticals that perform better anyway, so a raw average credits it with lift it did not cause.

Most of the effect runs through attributes you can see: LLM copy is longer, drops numbers, and reaches for aspirational verbs. That is a rewrite policy, not a verdict on language models.

## The loop: LLM proposes, TabPFN scores, instantly

`CopyCoach` fits once on the account. `revise` derives limits from the account's own winners (max words, keep the number, preferred tone), asks a language model for rewrites inside those limits, and scores the draft and every variant in one call. Set `ANTHROPIC_API_KEY` to use Claude; otherwise a deterministic stub stands in.

In [ ]:
from adlift.llm import get_llm
from adlift.loop import CopyCoach
from adlift.schema import AdCreative

coach = CopyCoach(account, model=CTRModel(backend=BACKEND), llm=get_llm()).fit()
print(f"fitted on {len(account)} creatives in {coach.fit_seconds:.2f}s with {coach.model.backend} / {coach.llm.name}")

draft = AdCreative(
    headline="Discover how Northwind can transform the way you handle manual reviews.",
    body="Join thousands of forward-thinking teams who have already made the switch.",
    cta_text="Begin your journey",
    campaign_id="new", author="llm",
    vertical="saas", placement="social_feed", device="mobile", audience="prospecting",
    daily_budget_usd=500, week_index=20,
)
result = coach.revise(draft, n_variants=6)
print(f"draft {result.draft.predicted_ctr * 100:.2f}%  ->  best {result.best.predicted_ctr * 100:.2f}%   (scored in {result.score_seconds:.2f}s)")
print(f"constraints: <= {result.constraints.max_words} words, tone {result.constraints.tone}, keep number {result.constraints.keep_numeric_claim}")
pd.DataFrame([v.to_dict() for v in result.variants])[["rank", "predicted_ctr", "lift_vs_draft", "word_count", "headline", "body", "cta_text"]]

## Read a banner, then score it

Ad creatives arrive as images. `banner_to_creative` hands the image to a vision-capable language model, gets the headline, body, CTA and layout attributes back as schema fields, and the same coach scores it. The placement context is yours to supply; a picture cannot tell you where it will run.

In [ ]:
from adlift.ingest import banner_to_creative

# Any local path or URL works. With ANTHROPIC_API_KEY the text is transcribed from the image.
BANNER = os.environ.get("ADLIFT_DEMO_BANNER", "")
if BANNER:
    creative = banner_to_creative(BANNER, llm=get_llm(), context={"device": "mobile", "placement": "social_feed", "vertical": "saas"})
    [scored] = coach.score([creative])
    print(creative.headline)
    print(f"predicted CTR {scored.predicted_ctr * 100:.2f}%  band [{scored.low * 100:.2f}, {scored.high * 100:.2f}]")
else:
    print("set ADLIFT_DEMO_BANNER to an image path or URL to run this cell")

## What to take away

- **TabPFN makes the loop interactive.** Fit in seconds, score a stack of rewrites in one call, no retraining when a new advertiser onboards. That is the property the whole design leans on.
- **The naive answer to "is LLM copy better" can have the wrong sign.** Adoption timing and vertical mix confound it. Adjust for context, not for the copy's own attributes.
- **The fix is a policy, not a verdict.** Most of the LLM penalty is length, missing numbers and tone. Constrain those and the model's wording is an asset.
- **Every number ships with its checks.** Placebo, overlap and estimator agreement are printed next to the estimate, and the synthetic account lets you verify the machinery recovers a known truth before you trust it on your own data.

The same pipeline is available as a CLI (`adlift demo`) and an MCP server (`adlift-mcp`); see the repository README.